<h1 align="center"><b>Homework Assignment 4 (100 points total)</b></h1>
<h3 align="center"><b>Assigned at the start of Module 9</b></h3>
<h3 align="center"><b>Due at the end of Module 11</b></h3><br>

# Q1 — Designing and Solving Your Own Reinforcement Learning Environment

## 30 points total

### [15 points] Part A – Design and Implement a Custom RL Environment

__Goal:__ Define and implement a small custom environment that can be solved using **Q-Learning** 

1. Use the [Gymnasium API structure](https://gymnasium.farama.org/introduction/create_custom_env/): 

```python
# starter code if you need it. Use documentation. Focus on design rather than syntax. 
import gymnasium as gym
from gymnasium import spaces

class CustomEnv(gym.Env):
    def __init__(self):
        super().__init__()
        self.observation_space = spaces.Discrete(...)
        self.action_space = spaces.Discrete(...)
        # define internal dynamics

    def reset(self, seed=None, options=None):
        # return initial state, info
        return state, {}

    def step(self, action):
        # define transitions, rewards, terminal condition
        return next_state, reward, terminated, truncated, {}
```
2. Choose one of these types of problems:

   - Grid-based navigation (agent finds goal, avoids obstacles)
   - Resource management (agent balances energy/reward trade-offs)
   - Sequential decision process (agent makes timed choices with delayed reward)

3. Your environment must have:

   - At least 8–12 discrete states
   - At least 2–4 actions
   - Clearly defined stochastic or deterministic transitions
   - A nontrivial reward structure (some decisions should require exploration)

4. Include a diagram of your environment (state transitions, rewards, or layout).
---

### [15 points] Part B – Apply Q-Learning and Analyze Results

- Implement Q-Learning
- Train for 1000+ episodes
- Plot:
  - Moving average of total episode reward
  - Final Q-table
  - Learning policy visualization

Analysis:
1. What did the agent learn?
   - Describe the final policy and what behaviors emerged
2. How does your reward design affect convergence?
   - Did the agent learn what you expected?
3. If you adjusted $\gamma$ or $\epsilon$, how did learning change?
4. What are the failure modes?
   - What would make your environment "harder" or "easier"?



## Q1 Response

### Design
- What skill should the agent learn?
  - A drone will learn to navigate through a city landscape
- What information does the agent need?
  - Position and velocity?
  - Current state of the system?
  - Historical data?
  - Partial or full observability?
- What actions can the agent take?
  - Discrete choices (move up/down/left/right)?
  - Continuous control (steering angle, throttle)?
  - Multiple simultaneous actions?
- How do we measure success?
  - Reaching a specific goal?
  - Minimizing time or energy?
  - Maximizing a score?
  - Avoiding failures?
- When should episodes end?
  - Task completion (success/failure)?
  - Time limits?
  - Safety constraints?

# Q2 — Particle Swarm Optimization (PSO): From-Scratch + Validation 💡
 
## 35 points total

---

### Goals
- Implement PSO from the velocity/position update equations and design sane hyperparameters.  
- Analyze convergence behavior and sensitivity to parameters $\omega, c_1, c_2$.  
- Validate your implementation against a standard PSO library.  
- Reflect on hybridizing PSO with Genetic Algorithm (GA) operators (theory only).  

---

### Objective Function (Rastrigin)
Use the Rastrigin function in $\mathbb{R}^d$ with box constraints $x_i \in [-5.12, 5.12]$:

$$
f(x) = 10d + \sum_{i=1}^{d} \left( x_i^2 - 10\cos(2\pi x_i) \right), \quad d = 30
$$

Known global optimum: $f(0) = 0$.  
This function is multimodal and separable—an excellent stress test for balancing exploration vs. exploitation.  
(*Optional:* You may also report results for $d \in \{10, 50\}$ in an appendix.)

---

### [10 points] Part A — Implement PSO from Scratch 

Implement canonical PSO with inertia using the following update equations:

$$
v_i(t+1) = \omega v_i(t) + c_1 r_1 \odot (pbest_i - x_i(t)) + c_2 r_2 \odot (gbest - x_i(t))
$$
$$
x_i(t+1) = x_i(t) + v_i(t+1)
$$

where $r_1, r_2 \sim U[0,1]^d$.  
After each update, clip positions to the bounding box.  

**Required features:**
- **Initialization:** $N = 60$ particles uniformly in $[-5.12, 5.12]^d$; initial velocities $= 0$.  
- Track each particle's **pbest** and global **gbest**.  
- Implement **velocity clamping** (e.g., $\|v\|_\infty \le 0.2 \times \text{range}$) or a **constriction factor** $\chi$ (cite which you used).  
- **Stopping condition:** 10,000 function evaluations **or** $f(gbest) \le 10^{-6}$.  
- Choose one **inertia schedule**:
  - Fixed $\omega$ (e.g., 0.7) with fixed $c_1, c_2$, or  
  - Linear decay $\omega: 0.9 \to 0.4$ over iterations.  
- **Randomness:** seed your RNG; run 20 independent trials.  

**Suggested defaults:** $\omega = 0.7, \; c_1 = c_2 = 1.6, \; v_{max} = 1.0$ per coordinate.

---

### [10 points] Part B — Convergence & Sensitivity Analysis
- Run experiments to analyze the **convergence behavior** and **sensitivity** of your PSO to $\omega, c_1, c_2$.  
- Report and discuss how different parameter settings affect convergence speed and final accuracy.  
- Plot **best-so-far vs. function evaluations** (mean $\pm$ 95% CI) across 20 trials.  
- Discuss **exploration–exploitation trade-offs** observed in your runs.  

**Deliverables:**
- Parameter sweep summary (tables or plots).  
- Key insights about sensitivity and stability.  

---

### [10 points] Part C — Validation Against a PSO Library
Validate your implementation using a PSO package such as `pyswarms` (`pip install pyswarms`).  

Keep the same:
- $N, d, \omega, c_1, c_2,$ and bounds.  
- Objective function and stopping criteria.  

**Report:**
- Final best $f$ (mean $\pm$ std over 20 trials) for *Your PSO* vs. *Library PSO*.  
- Fraction of trials achieving $f \le 10^{-3}$ (reliability).  
- Comparative plots (best-so-far curves).  

---

### [5 points] Part D — Theoretical Reflection

Reflect on **how PSO could be hybridized with GA operators** (e.g., crossover or mutation).  
- What advantages might this bring?  
- What risks or implementation challenges could arise?  


# Q3 — Explore $\beta$-VAE Trade-offs with KL Diagnostics on MNIST

## 35 points total

### Objective

Variational Autoencoders (VAEs) introduce a probabilistic latent space for generative modeling. Compared to a Vanilla Autoencoder, which only learns to reconstruct inputs, VAEs and $\beta$-VAEs learn a probabilistic latent space that supports sampling and generation. A $\beta$-VAE (beta-Variational Autoencoder) extends the standard VAE by introducing a weighting factor $\beta$ (hyperparameter) on the Kullback–Leibler (KL)-divergence term in the ELBO objective to balance reconstruction quality and latent regularization. 

$\mathcal{L}_{\beta\text{-VAE}} = \mathbb{E}_{q_\phi(z|x)}[-\log p_\theta(x|z)] + \beta \cdot \text{KL}(q_\phi(z|x) \parallel p(z))$

Increasing $\beta$ encourages more disentangled and interpretable latent features but can reduce reconstruction accuracy, revealing a key trade-off between structure and fidelity.

You will train a $\beta$-VAE with a convolutional encoder/decoder on MNIST, test some $\beta$ values, and compare their effects through plots and visuals.

### [5 points] Part A – Prepare Data

__Goal:__ Prepare the MNIST dataset for input to a convolutional $\beta$-VAE. Ensure all images are correctly normalized and shaped for a ConvNet-based model.

1. Use the MNIST handwritten digit dataset via ```torchvision.datasets.MNIST```.
2. Apply the following transformation via ```transforms.ToTensor()```
   - Normalize pixel values to the $[0,1]$ range. 
   - Reshape to (1, 28, 28) and treat as a single-channel grayscale image.
3. Verify image shape. 
4. (Optional) Use a subset of 10,000–20,000 samples from the training set to reduce training time.


### [10 points] Part B – Set up a $\beta$-VAE with a Convolutional Architecture

__Goal:__ Implement a convolutional $\beta$-VAE architecture suitable for MNIST. Understand how the encoder, decoder, and loss function interact in variational generative modeling.

Use the following convolutional architecture:
- **Encoder**: Fully connected layers that map input 28×28 pixels to a latent Gaussian distribution  
Conv2d $\rightarrow$ ReLU $\rightarrow$ Conv2d $\rightarrow$ ReLU $\rightarrow$ Flatten $\rightarrow$ FC layers to produce $\mu$ (mean of latent distribution) and $\log{\sigma^2}$ (log-variance of latent distribution)
- **Decoder**: Maps $z$ back to 28×28 pixels using a sigmoid output layer  
FC $\rightarrow$ Unflatten $\rightarrow$ ConvTranspose2d $\rightarrow$ ReLU $\rightarrow$ ConvTranspose2d $\rightarrow$ Sigmoid, that output should be a reconstructed image of (1, 28, 28)
- **Latent Prior**:  Use a smaller latent dimension (e.g., 2) if you wish to visualize the latent space; otherwise, use 10.
- **Loss**: Use binary cross-entropy (BCE) reconstruction loss and KL divergence  

$$
\mathcal{L}_{\beta\text{-VAE}} = \text{BCE}(x, \hat{x}) + \beta \cdot \text{KL}(q(z|x) \parallel \mathcal{N}(0, I)) \\
, \text{where  } z = \mu + \sigma \cdot \epsilon,\quad \epsilon \sim \mathcal{N}(0, I)
$$



### [10 points] Part C – Train β-VAE and Track Loss


__Goal:__ Train the model using different $\beta$ values and observe their effect on the balance between reconstruction and KL divergence.

- Choose at least two $\beta$ values greater than 0 (e.g., $\beta \in \{0.5, 1, 4\}$) or try linearly increasing values.
- Train for 10–15 epochs per setting.
- Use ```latent_dim = 10```
- Use Adam Optimizer (learning rate $lr = 1e‑3$, batch size = 128).
- For each epoch, log and print the following:
  - Total loss
  - Reconstruction loss (BCE)
  - KL divergence
- (Optional) Save trained model weights and per-epoch metrics for visualization in Part D.


### [10 points] Part D – Visualize and Interpret

__Goal:__ Assess how $\beta$ influences model performance in terms of reconstruction fidelity and sample quality.

1. For each trained model (for each $\beta$), do the following: 
   - Pick 5 test-set images and show original vs. reconstructed side-by-side.
   - Sample 5 random latent vectors from \( \mathcal{N}(0, I) \), decode them, and show the generated outputs.
2. Reflect on your results - the observed impact of $\beta$ on learning dynamics, reconstruction, and generative quality.
   - Which $\beta$ gives better reconstructions? Which gives more diverse or sharper samples?
   - How does increasing $\beta$ affect the KL divergence and reconstruction error?
   - Do you observe KL → 0 (posterior collapse)?
   - What are the risks of using too small or too large a $\beta$?